## Notebook 2 - Application Development

In [ ]:
# Allow nested event loops (needed so async agent calls work inside Jupyter)
import nest_asyncio
nest_asyncio.apply()

# Standard library
import os
import re
import sys
import io
import json
import time
import asyncio
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Type hints used by tool signatures (Annotated drives the JSON schema the agent sees)
from typing import Annotated, Optional, List, Literal

# Third-party: HTTP, data validation, database driver, env loading
import requests
from pydantic import Field
import psycopg
from dotenv import load_dotenv

# Agent Framework: LLM client + the @tool decorator that registers functions as callable tools
from agent_framework.openai import OpenAIChatClient
from agent_framework import tool

# Mem0: long-term memory layer for the agent (pgvector-backed in HorizonDB)
from mem0 import Memory

# UI + notebook display
import gradio as gr
from IPython.display import Markdown

In [ ]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_KEY        = os.environ["AZURE_OPENAI_KEY"]
AZURE_EMBED_DEPLOYMENT  = os.environ["AZURE_EMBED_DEPLOYMENT"]
AZURE_API_VERSION       = os.environ["AZURE_API_VERSION"]

DB_CONFIG = {
    # Note we are using the read-only endpoint for the agent as all queries
    # are read-only and is an example of offloading read workloads to a read replica
    "host":     os.environ["AZURE_PG_HOST"],
    "dbname":   os.environ["AZURE_PG_NAME"],
    "user":     os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port":     os.environ["AZURE_PG_PORT"],
    "sslmode":  os.environ.get("AZURE_PG_SSLMODE", "require"),
}

print(AZURE_OPENAI_DEPLOYMENT)
print(AZURE_EMBED_DEPLOYMENT)
print(AZURE_API_VERSION)

In [ ]:
@tool(description=(
    "BM25 full-text search over Washington case law opinions. "
    "Accepts plain keyword terms and optional filters by court level and minimum decision date. "
    "Returns the top matching cases ranked by BM25 relevance score. "
    "Use this first to surface canonical precedents by doctrine name or citation."
))
def keyword_case_search(
    query: Annotated[str, Field(
        description=(
            "Plain keyword terms for BM25 ranking over case opinions. "
            "Examples: 'municipal liability flooding', "
            "'surface water regrade grading', or 'street drainage sewer'."
        )
    )],
    # Optional metadata filter — restrict to one court tier. Useful when
    # the agent wants only binding precedent (Supreme Court) vs.
    # persuasive authority (Court of Appeals).
    court_level: Annotated[Optional[str], Field(
        default=None,
        description=(
            "Optional filter: 'Washington Supreme Court' "
            "or 'Washington Court of Appeals'. "
            "None = both courts."
        ),
    )] = None,
    # Optional metadata filter — exclude cases older than `min_year`.
    # Helpful for narrowing to modern doctrine while still allowing access
    # to historical canonical cases when min_year is None or very low.
    min_year: Annotated[Optional[int], Field(
        default=None,
        description="Optional filter: only return cases decided on or after this year."
    )] = None,
) -> str:
    # Diagnostic logging — printed to stdout so the Gradio UI's
    # "Tool Trace" panel can detect that this tool was invoked and display
    # its parameters. The trace parser keys off the literal string
    # "<tool_name> was called", so keep that line shape stable.
    print("keyword_case_search was called")
    print(f"  query={query!r}  court_level={court_level}  min_year={min_year}  limit=5")

    # Open a short-lived connection to Azure Database for PostgreSQL.
    # Using a context manager ensures the connection and cursor are closed
    # cleanly even if an exception is raised mid-query.
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        # pg_textsearch returns 0.0 for nonmatches, so ORDER BY alone can
        # fill the result set with unrelated rows. Bind the query once for
        # the projected score and again for the `< 0` match predicate, then
        # append any metadata parameters in SQL placeholder order.
        filters = ["(opinion <@> to_bm25query(%s, 'idx_cases_fts')) < 0"]
        params = [query, query]
        if court_level:
            filters.append("court_level = %s")
            params.append(court_level)
        if min_year:
            filters.append("decision_date >= %s")
            params.append(f"{min_year}-01-01")

        cur.execute(
            f"""
            SELECT id, name, decision_date, court_level,
                   opinion <@> to_bm25query(%s, 'idx_cases_fts') AS score,
                   opinion
            FROM cases
            WHERE {' AND '.join(filters)}
            ORDER BY score
            LIMIT 5;
            """,
            tuple(params),
        )
        rows = cur.fetchall()

    if not rows:
        return "No matches"

    # `<@>` returns negative BM25 scores. Results are sorted ascending, so
    # lower (more negative) scores represent stronger lexical matches.
    lines = []
    for case_id, name, decision_date, level, score, opinion in rows:
        snippet = (opinion or "").replace("\n", " ")[:300]
        lines.append(
            f"{case_id} | {decision_date} | {level} | score={score:.3f} | "
            f"{name}: {snippet}..."
        )
    return "\n".join(lines)


In [ ]:
print("// Test Call of keyword_case_search //")
print(keyword_case_search("surface water OR flooding OR drainage"))

In [ ]:
# Build the underlying chat client.
#
# Despite the class name `OpenAIChatClient`, this is configured to talk
# to Azure OpenAI by passing `azure_endpoint`. The Agent Framework SDK
# routes requests to your Azure OpenAI deployment using:
#   - model:           the Azure OpenAI deployment name (NOT the base
#                      model name like "gpt-5"). This is the name you
#                      gave the deployment in Azure AI Foundry / the
#                      Azure portal.
#   - azure_endpoint:  the resource endpoint, e.g.
#                      https://<your-resource>.openai.azure.com/
#   - api_key:         the resource key. In production prefer Azure AD /
#                      Managed Identity over a static key.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Turn the chat client into an Agent.
#
# `as_agent(...)` wraps the chat client in an agent loop that:
#   1. Sends the user message + the system `instructions` to the LLM.
#   2. Advertises the `tools` list to the LLM as callable functions
#      (the JSON schema is auto-generated from each @tool-decorated
#      function's signature and Annotated[..., Field(description=...)]
#      metadata — that's why those descriptions matter).
#   3. If the LLM responds with a tool call, the framework executes the
#      Python function, feeds the result back into the conversation,
#      and lets the model continue reasoning.
#   4. Repeats until the model produces a final natural-language answer.
#
# `instructions` is the system prompt. Keep it short and directive:
#   - Tells the model its role ("legal assistant").
#   - Tells it the desired output shape (case names + why relevant).
#   - "Run each tool only once" discourages the model from re-calling
#     the same tool with slightly tweaked arguments, which wastes
#     latency and token budget without improving recall.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Run every tool only once."
    ),
    tools=[keyword_case_search],
)

# The user question. Notice we do NOT pre-extract keywords or hand the
# tool a search string directly — that's the agent's job. The LLM will
# read this natural-language prompt, decide which terms to search for
# (e.g., "surface water", "drainage", "common enemy doctrine"), and
# call keyword_case_search with arguments of its own choosing.
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the city "
    "redid the drainage on the road."
)

# Run the agent. `agent.run(...)` is async, so we `await` it. Because
# we called `nest_asyncio.apply()` at the top of the notebook, awaiting
# at the top level of a cell works inside Jupyter.
#
# While this runs, watch the cell output: the `print(...)` lines inside
# keyword_case_search (e.g., "keyword_case_search was called") will
# appear, which is how you confirm the agent actually invoked the tool
# rather than hallucinating an answer from prior knowledge.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# `result.text` is the agent's final natural-language answer (after any
# tool calls have been folded in). We render it as Markdown so case
# names, bullets, and formatting display nicely in the notebook.
print("")
print("// Agent Response: //")
Markdown(result.text)


In [ ]:
# We validate against this set so a typo from the LLM produces a clean error
# message instead of silently returning zero rows.
VALID_COURT_LEVELS = {"Washington Supreme Court", "Washington Court of Appeals", "Both"}

@tool(description=(
    "Semantic vector search over Washington case opinions using DiskANN with "
    "Advanced Filtering enabled (filters are pushed into the index traversal). "
    "Returns the top 5 cases ranked by cosine similarity to the query embedding. "
    "Both court_level and min_year are REQUIRED — pass court_level='Both' to search "
    "across both courts, and pass min_year=1900 if you do not want a date cutoff. "
    "This tool runs independently of keyword_case_search; the caller is expected to "
    "take the UNION of ids from both tools before passing them to precedent_graph_search."
))
def semantic_case_search(
    # The natural-language query. Unlike Tool 1, no special syntax —
    # the embedding model captures meaning, not literal tokens. So
    # "developer regraded lot causing flooding" works as well as a
    # carefully crafted boolean expression would in BM25.
    query_text: Annotated[str, Field(
        description="Natural language query describing the fact pattern or legal issue."
    )],
    # REQUIRED court filter. The model must pick one of the three
    # values. "Both" is the explicit way to skip filtering by court so
    # the model never has to omit the argument.
    court_level: Annotated[Literal["Washington Supreme Court", "Washington Court of Appeals", "Both"], Field(
        description=(
            "REQUIRED court filter. Use 'Washington Supreme Court' for binding precedent only, "
            "'Washington Court of Appeals' for that court only, or 'Both' to search across both courts."
        ),
    )],
    # REQUIRED year-floor filter. The model must pass an integer; use
    # 1900 (or earlier) to effectively disable the cutoff while still
    # giving DiskANN's advanced filtering a predicate to work with.
    min_year: Annotated[int, Field(
        description=(
            "REQUIRED minimum decision year. Only cases decided on or after Jan 1 of this year "
            "are returned. Pass 1900 to effectively disable the cutoff."
        )
    )],
) -> str:
    # Diagnostic trace line — the Gradio UI's Tool Trace panel keys off
    # the literal "<tool_name> was called" string, so keep that shape.
    print("semantic_case_search was called")
    print(
        f"  query_text={query_text!r}  court_level={court_level}  "
        f"min_year={min_year}  limit=5  advanced_filtering=ALWAYS_ON"
    )

    # Validate against the whitelist. Returning early with a clear
    # message helps the LLM self-correct on its next turn.
    if court_level not in VALID_COURT_LEVELS:
        return f"Invalid court_level {court_level!r}. Use one of: {sorted(VALID_COURT_LEVELS)}"

    # Short-lived autocommit connection — SET statements affect only
    # this session, so a fresh connection per call gives us a clean
    # GUC state every time.
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        # ──────────────────────────────────────────────────────────────
        # DiskANN advanced-filtering GUCs (session-level) — ALWAYS ON.
        # These tell the DiskANN index to evaluate the WHERE predicates
        # DURING graph traversal rather than as a post-filter. The
        # values below are sensible defaults for this lab; tune per
        # workload in production.
        #
        #   enable_filter_hook    – master switch for advanced filtering.
        #   selectivity_min/_threshold – when to engage filtered traversal
        #         based on estimated predicate selectivity (fraction of
        #         rows expected to match). 0.0..1.0 means "always try".
        #   filtering_beta        – tradeoff between recall and latency
        #         when navigating the filtered subgraph. Higher = more
        #         exploration = better recall, slightly slower.
        #   l_value_is            – search list size for the in-memory
        #         portion of the index. Larger = better recall, slower.
        # ──────────────────────────────────────────────────────────────
        cur.execute("SET diskann.enable_filter_hook = 'true';")
        cur.execute("SET diskann.selectivity_min = '0.0';")
        cur.execute("SET diskann.selectivity_threshold = '1.0';")
        cur.execute("SET diskann.filtering_beta = 0.85;")
        cur.execute("SET diskann.l_value_is = 300;")

        # ──────────────────────────────────────────────────────────────
        # Build the WHERE clause. We use NAMED placeholders (%(name)s)
        # so we can pass a single params dict.
        #
        # `opinions_vector IS NOT NULL` excludes any case where the
        # embedding wasn't generated (defensive — shouldn't happen in
        # this lab, but cheap insurance).
        #
        # court_level filter is added unless the model passed "Both"
        # (the explicit "no court filter" sentinel). min_year is always
        # applied — pass 1900 for "no cutoff".
        # ──────────────────────────────────────────────────────────────
        where = [
            "opinions_vector IS NOT NULL",
            "decision_date >= make_date(%(min_year)s, 1, 1)",
        ]
        params = {
            "q": query_text,
            "lim": 5,
            "min_year": min_year,
            # The embedding deployment name (e.g. "text-embedding-3-small")
            # is what azure_openai.create_embeddings() needs to know which
            # Azure OpenAI deployment to call. The endpoint + key were
            # configured back in Notebook 1 via azure_ai.set_setting().
            "embed_deployment": "lab-embedding",
        }

        if court_level != "Both":
            where.append("court_level = %(court_level)s")
            params["court_level"] = court_level
        
        embed_sql = (
            "azure_openai.create_embeddings(%(embed_deployment)s, %(q)s)::vector"
        )

        # ──────────────────────────────────────────────────────────────
        # The core semantic-search query.
        #
        #   `opinions_vector <=> <query_vector>`  – cosine DISTANCE
        #       (NOT similarity). Smaller = more similar. 0.0 is a
        #       perfect match; 2.0 is opposite. Other pgvector operators:
        #         <->  L2 distance
        #         <#>  negative inner product
        #
        #   ORDER BY distance ASC + LIMIT 5  – this is the pattern the
        #   DiskANN planner recognizes to engage the index. Without an
        #   ORDER BY on the distance expression it would fall back to a
        #   sequential scan, which is dramatically slower on large
        #   corpora.
        # ──────────────────────────────────────────────────────────────
        cur.execute(
            f"""
            SELECT id, name, decision_date, court_level,
                   opinions_vector <=> {embed_sql} AS distance,
                   opinion
            FROM public.cases
            WHERE {' AND '.join(where)}
            ORDER BY distance ASC
            LIMIT %(lim)s;
            """,
            params,
        )
        rows = cur.fetchall()

    # Nothing matched.
    if not rows:
        return "No matches"

    # ──────────────────────────────────────────────────────────────────
    # Format the response. Same pipe-delimited shape as Tool 1 so the
    # `parse_ids` helper used in later cells can extract case ids from
    # either tool's output with the same regex (^(\d+)\s+\|).
    # Snippets are flattened to one line and truncated to 300 chars to
    # control LLM context size.
    #
    # NOTE: This output contains only the 5 vector-search hits. To feed
    # Tool 3, take the UNION of these ids with Tool 1's ids — see the
    # test cell below for the pattern.
    # ──────────────────────────────────────────────────────────────────
    lines = []
    for case_id, name, decision_date, level, dist, opinion in rows:
        snippet = (opinion or "").replace("\n", " ")[:300]
        dist_str = "n/a" if dist is None else f"{dist:.4f}"
        lines.append(
            f"{case_id} | {decision_date} | {level} | distance={dist_str} | "
            f"{name}: {snippet}..."
        )

    return "\n".join(lines)


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Step 1: Tool 1 (BM25 keyword search)
# ─────────────────────────────────────────────────────────────────────
# Supply plain legal-doctrine terms. pg_textsearch ranks documents by
# BM25 relevance; it doesn't natively implement phrase/proximity syntax.
# Filtering to "Washington Supreme Court" keeps results to binding
# precedent only; min_year=1900 effectively disables the date floor.
print("// Step 1: Keyword anchors (BM25) - returns up to 5 cases //")
kw = keyword_case_search(
    "surface water drainage stormwater",
    court_level="Washington Supreme Court",
    min_year=1900,
)
print(kw)

# Each result row from our tools starts with "<case_id> | ..." (pipe
# delimited). This regex pulls just the leading integer id off each
# line so we can build a de-duplicated input set for Tool 3 later.
kw_ids = []
for line in kw.splitlines():
    m = re.match(r"^(\d+)\s+\|", line)
    if m:
        kw_ids.append(int(m.group(1)))

print("")

# ─────────────────────────────────────────────────────────────────────
# Step 2: Tool 2 (semantic vector search via pgvector + DiskANN)
# ─────────────────────────────────────────────────────────────────────
# This time we pass a natural-language fact pattern. Tool 2 will embed
# it with Azure OpenAI and run a cosine-similarity search against the
# cases.opinion_embedding column. court_level and min_year are REQUIRED
# args, so we use the "no-op" sentinel values:
#   - court_level="Both"  -> do not filter by court
#   - min_year=1900       -> effectively no date cutoff
# Even when "no-op", these values still give DiskANN's advanced
# filtering a predicate to honor during index traversal.
print("// Step 2: Semantic expansion (DiskANN, Advanced Filtering on) - returns up to 5 cases //")
flagship_prompt = (
    "I have a client whose property keeps flooding after the city redid the drainage on the road."
)
sem = semantic_case_search(
    flagship_prompt,
    court_level="Both",   # "Both" = search across both courts (required arg, no None allowed)
    min_year=1900,        # 1900 = effectively no cutoff (required arg)
)
print(sem)

# Same regex parser as Step 1, applied to Tool 2's output to pull its
# up-to-5 case ids.
sem_ids = []
for line in sem.splitlines():
    m = re.match(r"^(\d+)\s+\|", line)
    if m:
        sem_ids.append(int(m.group(1)))

print("")

# ─────────────────────────────────────────────────────────────────────
# Step 3: Build the input set for Tool 3 (citation-graph expansion)
# ─────────────────────────────────────────────────────────────────────
# Tool 3 will consume the UNION of ids from Tool 1 + Tool 2 (up to 10
# unique cases). The set() | set() expression is Python's set-union
# operator, which automatically dedupes any case that was returned by
# both tools — those are the strongest "this is definitely on point"
# candidates. sorted(...) just gives stable output for readability.
print("// Step 3 preview: union of Tool 1 + Tool 2 ids -> inputs for Tool 3 //")
input_ids = sorted(set(kw_ids) | set(sem_ids))
print(f"Tool 1 ids ({len(kw_ids)}): {kw_ids}")
print(f"Tool 2 ids ({len(sem_ids)}): {sem_ids}")
print(f"Union  ids ({len(input_ids)}): {input_ids}")


In [ ]:
# Step 1: Build a fresh Azure OpenAI chat client.
# OpenAIChatClient + azure_endpoint routes through your Azure OpenAI deployment.
# `model` here is the *deployment name* from Azure AI Foundry, not the base model id.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Step 2: Wrap the chat client in an agent that knows about BOTH retrieval tools.
# `as_agent()` snapshots the tools list at construction time, so we rebuild the
# agent each time we add a new tool. The instructions are the agent's system
# prompt: keep them short and behavioral ("respond with...", "run each tool only once").
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Run each tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search
    ],
)

# Step 3: Same Seattle flooding scenario as the Part 3.3.1 smoke test, so you can
# compare the agent's narrative answer against the raw rows we printed earlier.
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the city "
    "redid the drainage on the road."
)

# Step 4: Run the agent. Under the hood Agent Framework will:
#   (a) send the prompt + tool schemas to Azure OpenAI,
#   (b) execute any tool calls the model requests (printing the "was called" lines),
#   (c) feed tool outputs back into the conversation,
#   (d) loop until the model returns a final natural-language answer.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# Step 5: Render the agent's final answer as Markdown so case names, bullets,
# and emphasis show up nicely in the notebook output.
print("")
print("// Agent Response: //")
Markdown(result.text)


In [ ]:
@tool(description=(
    "Traverse the Washington case citation graph stored in Apache AGE (graph name: case_graph). "
    "Edges are (a)-[:REF]->(b) meaning 'a cites b'. "
    "Given input case ids (from keyword_case_search and semantic_case_search), "
    "returns the TOP 10 most relevant related precedent cases (by centrality score) via a "
    "2-hop bidirectional citation traversal (follows both 'cites' and 'cited-by' edges), "
    "enriched with metadata from the relational cases table. "
    "Washington Supreme Court cases are always boosted in the ranking as binding precedent."
))
def precedent_graph_search(
    input_case_ids: Annotated[List[int], Field(
        description=(
            "List of input case ids from Tool 1 and/or Tool 2. "
            "Pass the union of ids from both tools for best coverage."
        )
    )],
) -> str:
    # Cap the number of related cases returned, so Tool 4 (extract) doesn't
    # get flooded with low-relevance hits. We rank by `score` (see below)
    # and take the top N.
    TOP_K = 10

    # Court-weight boost for Washington Supreme Court (binding precedent).
    # Always applied — there is no opt-out, because binding precedent
    # should always outrank persuasive Court of Appeals authority when
    # path counts are otherwise comparable.
    SUPREME_COURT_BOOST = 2

    # Fixed traversal parameters — kept inside the function so the LLM
    # doesn't have to think about them and the lab demo stays consistent.
    #
    #   HOPS = 2  → up to cite-of-cite expansion. 1 would be direct
    #               citations only (often too narrow). 3 starts pulling
    #               in weakly-related cases and dramatically increases
    #               result size.
    #   DIRECTION = "both" → follow both outgoing edges (input cites prior
    #               authority) AND incoming edges (later cases cite the
    #               input). The union gives the most complete picture of
    #               the line of authority around the input cluster.
    HOPS = 2

    print("precedent_graph_search was called")
    print(f"  inputs={len(input_case_ids)}  direction=both  hops={HOPS}  top_k={TOP_K}  supreme_boost={SUPREME_COURT_BOOST}")

    if not input_case_ids:
        return "No input_case_ids provided."

    # Graph stores case_id as text property, created from temp_cases.data->>'id'
    # Your helper functions store it as text, so we compare using string literals.
    input_literals = ", ".join([f"'{int(i)}'" for i in sorted(set(input_case_ids))])

    # Build Cypher fragments for both directions of the REF edge.
    # REF direction: (a)-[:REF]->(b) means "a cites b"
    #   - "cites":    input -> authority          (outgoing from input)
    #   - "cited_by": later_case -> input         (incoming to input,
    #                                              traversed by matching
    #                                              (t)-[:REF*]->(s))
    # We always run both and aggregate path counts together.
    cypher_queries = [
        f"""
            SELECT * FROM cypher('case_graph', $$
                MATCH (s:case)-[:REF*1..{HOPS}]->(t:case)
                WHERE s.case_id IN [{input_literals}]
                RETURN t.case_id AS related_id, count(*) AS paths
            $$) AS (related_id agtype, paths agtype);
        """,
        f"""
            SELECT * FROM cypher('case_graph', $$
                MATCH (t:case)-[:REF*1..{HOPS}]->(s:case)
                WHERE s.case_id IN [{input_literals}]
                RETURN t.case_id AS related_id, count(*) AS paths
            $$) AS (related_id agtype, paths agtype);
        """,
    ]

    # Execute graph traversal and aggregate path counts.
    related_paths = {}  # related_id(int) -> paths(int)
    try:
        with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
            # AGE session setup            
            cur.execute("SET search_path = public, ag_catalog, \"$user\";")

            for q in cypher_queries:
                cur.execute(q)
                for related_id_ag, paths_ag in cur.fetchall():
                    # AGE returns agtype; psycopg gives it as Python str like '"123"' or '123'
                    rid_str = str(related_id_ag).strip('"')
                    if not rid_str.isdigit():
                        continue
                    rid = int(rid_str)

                    # Skip inputs in the related set; we only want expansions
                    if rid in input_case_ids:
                        continue

                    p = int(str(paths_ag).strip('"')) if str(paths_ag).strip('"').isdigit() else 1
                    related_paths[rid] = related_paths.get(rid, 0) + p

    except Exception as e:
        return f"Graph query failed. Is case_graph created and populated? Error: {e}"

    if not related_paths:
        return "No related cases found from the citation graph for the provided inputs."

    related_ids = list(related_paths.keys())

    # Enrich from relational table so Tool 4 can immediately extract from opinion text.
    # Also compute a simple ranking score: paths (centrality proxy) + court weighting.
    try:
        with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
            cur.execute(
                """
                SELECT id, name, decision_date, court_level, opinion
                FROM public.cases
                WHERE id = ANY(%s);
                """,
                (related_ids,)
            )
            rows = cur.fetchall()
    except Exception as e:
        return f"Failed to fetch case metadata from relational table. Error: {e}"

    # Court weighting: WA Supreme Court always gets the binding-precedent
    # boost; Court of Appeals (and anything else) gets 0.
    def court_weight(level: str) -> int:
        if level == "Washington Supreme Court":
            return SUPREME_COURT_BOOST
        return 0

    # ──────────────────────────────────────────────────────────────────
    # Score = paths + court_weight
    #
    #   paths        = count of distinct multi-hop citation paths from
    #                  any input to this case (a centrality proxy — more
    #                  paths = the case sits closer to the input cluster
    #                  in the citation graph).
    #   court_weight = +2 if Washington Supreme Court, else 0. Coarse
    #                  boost for binding precedent over Court of Appeals.
    #                  Always applied (no opt-out).
    #
    # Ties on `score` are broken by raw `paths`.
    # ──────────────────────────────────────────────────────────────────
    ranked = []
    for case_id, name, decision_date, court_level, opinion in rows:
        paths = related_paths.get(case_id, 0)
        score = paths + court_weight(court_level)
        snippet = (opinion or "").replace("\n", " ")[:240]
        ranked.append((score, paths, case_id, decision_date, court_level, name, snippet))

    ranked.sort(key=lambda x: (x[0], x[1]), reverse=True)

    # Keep only the top-K highest-scoring related cases. This bounds the
    # payload size handed off to Tool 4 (extract) and to the LLM context
    # window, while still keeping the most central / binding precedents.
    total_related = len(ranked)
    ranked = ranked[:TOP_K]

    print(f"  related_count_total={total_related}  returned_top_k={len(ranked)}")

    # Return a parseable, stable text format:
    # - first section: ids only (for Tool 4 handoff) — already trimmed to top-K
    # - second section: human-readable justification
    top_ids = [str(r[2]) for r in ranked]
    lines = []
    lines.append("RELATED_CASE_IDS: " + ", ".join(top_ids))
    lines.append("")
    lines.append(f"Related cases (top {len(ranked)} of {total_related}, ranked by score):")
    for score, paths, case_id, decision_date, court_level, name, snippet in ranked:
        lines.append(
            f"{case_id} | {decision_date} | {court_level} | paths={paths} | score={score} | {name}: {snippet}..."
        )

    return "\n".join(lines)


In [ ]:
# Helper: extract case ids from a tool's text output.
#
# Tools 1 and 2 both return rows shaped like:
#     "12345 | 1953-04-02 | Washington Supreme Court | score=... | Name: snippet..."
# A tiny regex grabs the leading integer on each row so we can build the
# de-duplicated input set that Tool 3 (the citation graph) expects.
def parse_ids(tool_output: str) -> list[int]:
    ids = []
    for line in tool_output.splitlines():
        m = re.match(r"^(\d+)\s+\|", line)
        if m:
            ids.append(int(m.group(1)))
    return ids

# Step 1: Tool 1 (BM25 keyword search).
# We narrow to Washington Supreme Court because we want binding-precedent
# anchors to seed the graph traversal. Use plain keyword terms for
# pg_textsearch BM25 ranking.
kw_out = keyword_case_search(
    "surface water drainage stormwater",
    court_level="Washington Supreme Court",
    min_year=1900,
)

# Step 2: Tool 2 (semantic / vector search with DiskANN advanced filtering).
# Natural-language fact pattern instead of keywords. "Both" courts and
# min_year=1900 are passed explicitly so DiskANN's advanced filtering
# always has predicates to push down (the lab demo is meant to exercise
# that code path on every call).
sem_out = semantic_case_search(
    "city changed road drainage",
    court_level="Both",   # "Both" = search across both courts (required arg)
    min_year=1900,        # 1900 = effectively no cutoff (required arg)
)

# Step 3: Build the input set for Tool 3.
# Union of ids from Tools 1 + 2, de-duplicated and sorted for a stable
# trace. This list is exactly what the agent will compute on its own
# in Part 3.4.2 when it gets all three tools at once.
input_ids = sorted(set(parse_ids(kw_out) + parse_ids(sem_out)))
print("Input ids:", input_ids)

# Step 4: Tool 3 (Apache AGE citation graph traversal).
# Expands the input cluster via 2-hop bidirectional [:REF] edges and
# returns the top 10 related precedents, ranked by paths + court boost.
# Watch for: precedent_graph_search trace line, the parameters echo,
# related_count_total vs returned_top_k, and the RELATED_CASE_IDS header
# (which is the parseable handoff for Tool 4 later in the notebook).
print("")
print("// Tool 3: Graph traversal //")
graph_out = precedent_graph_search(input_case_ids=input_ids)
print(graph_out)


In [ ]:
# Step 1: Build a fresh Azure OpenAI chat client.
# Same pattern as Part 3.3.2: OpenAIChatClient + azure_endpoint routes through
# your Azure OpenAI deployment, and `model` is the *deployment name* from
# Azure AI Foundry.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Step 2: Rebuild the agent with ALL THREE retrieval tools.
# `as_agent()` snapshots the tools list at construction time, so we
# re-create the agent each time we add a new tool. The LLM now has to
# decide on its own to: (a) call keyword + semantic to gather candidate
# ids, (b) take the union, (c) pass them to precedent_graph_search as
# `input_case_ids`. The tool descriptions we wrote earlier are what teach
# it that chaining.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Run each tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search
    ],
)

# Step 3: Same Seattle flooding scenario, so the
# narrative answer below is directly comparable to the raw rows you
# printed by hand a few cells ago.
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the city redid the drainage on the road."
)

# Step 4: Run the agent. Under the hood Agent Framework will:
#   (a) send the prompt + all three tool schemas to Azure OpenAI,
#   (b) execute each tool call the model requests (printing the
#       "was called" trace lines from inside each @tool function),
#   (c) feed each tool's output back into the conversation so the next
#       tool call (and the final answer) can use it,
#   (d) loop until the model returns a natural-language answer.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# Step 5: Render the agent's final answer as Markdown. Expect case names
# pulled from all three tools, with Washington Supreme Court precedents
# (boosted by Tool 3) doing most of the rule-statement work.
print("")
print("// Agent Response: //")
Markdown(result.text)


In [ ]:
@tool(description=(
    "Extract structured legal facts from one or more case opinions using azure_ai.extract. "
    "Returns JSON per case with compact fields (holding and issues by default). "
    "Use after the graph tool to turn raw opinions into brief-ready structured records."
))
def case_analyst_extract(
    case_ids: Annotated[List[int], Field(
        description=(
            "List of case ids to analyze (typically from precedent_graph_search's RELATED_CASE_IDS)."
        )
    )]
) -> str:

    # The chat deployment name (set in your .env) is the model that
    # azure_ai.extract will call from inside the database. It is also
    # what the agent uses for its own reasoning, but here we are using
    # it specifically for structured extraction over case opinions.
    chat_deployment = "lab-chat"

    # Trace lines for the Gradio "Tool Trace" panel later in the notebook.
    print("case_analyst_extract was called")
    print(f"  case_ids={len(case_ids)}  chat_deployment={chat_deployment}")

    # Defensive: if Tool 3 returned no related cases, return an empty
    # JSON envelope so downstream code does not blow up parsing it.
    if not case_ids:
        return "CASE_BRIEFS_JSON: []"

    # The structured fields we want azure_ai.extract to pull out of every
    # opinion. Keep this list short and stable: the agent's downstream
    # prompt is written assuming these exact keys exist on every extracted card.
    fields = ["holding", "issues", "statutes_cited", "disposition"]

    # Emit the entity fields the agent is extracting (parsed by the UI trace panel)
    print(f"  fields={','.join(fields)}")

    # Demo-latency cap: truncate each opinion to ~1200 chars before
    # sending it to azure_ai.extract. In production you would chunk and
    # aggregate, not truncate.
    max_opinion_chars = 1200

    # Step 1: Pull the case rows we need (id, name, date, court, opinion)
    # in a single relational query. ANY(%s) lets us pass the whole id
    # list as one parameter instead of building a comma-joined string.
    # CREATE EXTENSION IF NOT EXISTS azure_ai is idempotent and just
    # makes the extension safe to call from this connection.
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS azure_ai;")
        cur.execute(
            """
            SELECT id, name, decision_date, court_level, opinion
            FROM public.cases
            WHERE id = ANY(%s);
            """,
            (case_ids,),
        )
        rows = cur.fetchall()

    # If somehow none of the requested ids exist in the table (e.g., a
    # bad handoff from Tool 3), short-circuit with an empty envelope.
    if not rows:
        return "CASE_BRIEFS_JSON: []"

    # Step 2: Extract in parallel - each thread opens its own connection
    # so the LLM calls run concurrently instead of sequentially.
  
    extract_sql = "SELECT azure_ai.extract(%s::text, %s::text[], %s::text);"
    extract_model = chat_deployment

    # Worker that runs inside each thread: one DB connection per thread
    # (psycopg connections are NOT thread-safe to share), one azure_ai.extract
    # call per case. Wrapping the call in try/except means a single bad
    # opinion will not nuke the whole batch.
    def _extract_one(row):
        case_id, name, decision_date, court_level, opinion = row
        short_opinion = (opinion or "")[:max_opinion_chars]
        try:
            with psycopg.connect(**DB_CONFIG, autocommit=True) as c2, c2.cursor() as cur2:
                cur2.execute(
                    extract_sql,
                    (short_opinion, fields, extract_model),
                )
                # azure_ai.extract returns JSON; psycopg already decodes
                # JSONB columns to a Python dict, so no json.loads needed.
                extracted = cur2.fetchone()[0]
        except Exception as e:
            extracted = {"error": "extraction_failed", "reason": str(e)}

        # One JSON card per case, in the exact shape the agent's downstream
        # prompt expects.
        card = {
            "id": case_id,
            "name": name,
            "decision_date": str(decision_date),
            "court_level": court_level,
            "extracted": extracted,
        }
        return card

    # Fan out: one worker per row so all extractions run concurrently.
    # as_completed() streams results back in finish order, which is also
    # the natural order to emit the per-case trace lines for the UI.
    with ThreadPoolExecutor(max_workers=len(rows)) as pool:
        futures = {pool.submit(_extract_one, r): r[0] for r in rows}
        results = []
        for fut in as_completed(futures):
            card = fut.result()
            results.append(card)
            # Emit per-case completion line for the UI trace panel
            cname = (card.get("name") or "").replace("|", "/")[:60]
            print(f"  extracted_case={card.get('id')}|{cname}", flush=True)

    # Step 3: Re-sort the results into the original input id order so the
    # JSON output is deterministic (the parallel pool returns them in
    # whatever order finished first, which is not useful for the agent).
    id_order = {cid: i for i, cid in enumerate(case_ids)}
    results.sort(key=lambda c: id_order.get(c["id"], 1_000_000))

    # Stable header + indented JSON body. The "CASE_BRIEFS_JSON:" prefix
    # is the parseable handoff marker that the agent (and the Gradio UI)
    # looks for when picking the extracted briefs out of this tool's output.
    return "CASE_BRIEFS_JSON: " + json.dumps(results, indent=2, default=str)


In [ ]:
# Step 1: Build a fresh Azure OpenAI chat client.
# Same pattern as Parts 3.9 and 3.12: OpenAIChatClient + azure_endpoint
# routes through your Azure OpenAI deployment. `model` is the *deployment
# name* from Azure AI Foundry, not the base model id.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Step 2: Rebuild the agent with ALL FOUR tools.
# `as_agent()` snapshots the tools list at construction time, so we
# re-create the agent each time we add a new tool.
#
# The LLM now has to chain: keyword + semantic to gather candidate ids ->
# union them -> precedent_graph_search to expand via the citation graph ->
# parse the RELATED_CASE_IDS line out of Tool 3's output -> case_analyst_extract
# to pull structured holdings/issues/statutes/disposition out of each
# opinion. The tool descriptions you wrote earlier are what teach the
# model that whole sequence.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with holdings, statutes, issues, and dispositions. Run every tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract,
    ],
)

# Step 3: Same Seattle flooding scenario as earlier in the notebook,
# so the narrative answer below is directly comparable to the earlier
# agent runs. The interesting question is: does the answer get
# qualitatively better now that Tool 4 is in the mix?
user_query = (
    "I have a client in Seattle, WA whose property keeps flooding after the city redid the drainage on the road."
)

# Step 4: Run the agent. Under the hood Agent Framework will:
#   (a) send the prompt + all four tool schemas to Azure OpenAI,
#   (b) execute each tool call the model requests (printing the
#       "was called" trace lines from inside each @tool function;
#       Tool 4 also prints one "extracted_case=..." line per case as
#       the ThreadPoolExecutor workers finish),
#   (c) feed each tool's output back into the conversation so the next
#       tool call (and the final answer) can use it,
#   (d) loop until the model returns a natural-language answer.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# Step 5: Render the agent's final answer as Markdown. Expect a memo
# that quotes actual holdings, lists the issues each court decided, and
# cites statutes when Tool 4's `statutes_cited` field surfaced any.
# That qualitative jump versus Part 3.4.2 is what Tool 4 buys you.
print("")
print("// Agent Response: //")
Markdown(result.text)


In [ ]:
# Note the word "OPTIONAL" in the description plus the explicit list of
# trigger words (rain, weather, water, storms, flooding, drainage,
# precipitation). That sentence is the agent's user manual for *when*
# to call this tool. The LLM reads it and decides whether the user's
# question warrants pulling weather evidence at all.
@tool(description=(
    "Retrieve historical precipitation data for a location and date range to support "
    "weather-related legal claims (rain, flooding, storm damage, drainage disputes). "
    "Uses the Open-Meteo Archive API. This tool is OPTIONAL - only call it when the "
    "legal question involves rain, weather, water, storms, flooding, drainage, or precipitation."
))
def get_weather_evidence(
    # latitude / longitude / start_date are REQUIRED so the LLM has to
    # commit to a specific location and date before calling. end_date is
    # optional and defaults to start_date, which lets the model ask for
    # a single day without having to think about a range.
    latitude: Annotated[float, Field(
        description="Latitude of the location (WGS84). Example: 47.6062 for Seattle."
    )],
    longitude: Annotated[float, Field(
        description="Longitude of the location (WGS84). Example: -122.3321 for Seattle."
    )],
    start_date: Annotated[str, Field(
        description="Start date in YYYY-MM-DD format."
    )],
    end_date: Annotated[Optional[str], Field(
        default=None,
        description="End date in YYYY-MM-DD format. Defaults to start_date (single day)."
    )] = None,
) -> str:
    # Trace lines for the Gradio "Tool Trace" panel later in the notebook.
    print("get_weather_evidence was called")
    print(f"  lat={latitude}  lon={longitude}  start={start_date}  end={end_date or start_date}")

    # Single-day query convenience: if the model only supplied a start
    # date, treat the range as that one day.
    if end_date is None:
        end_date = start_date

    # Step 1: Build the Open-Meteo Archive API request.
    # Free, no API key required. We only ask for daily precipitation_sum
    # to keep the payload small and the LLM context window happy.
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum",
        "timezone": "UTC",
    }

    # Step 2: Fire the request with a 10-second timeout so a slow network
    # cannot stall the entire agent run. Any failure (timeout, HTTP error,
    # bad JSON) is caught and surfaced as a WEATHER_EVIDENCE_ERROR string
    # rather than a Python exception - the agent can then carry on with
    # what it already has from Tools 1-4.
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return f"WEATHER_EVIDENCE_ERROR: Failed to retrieve weather data: {e}"

    # Step 3: Parse the parallel arrays Open-Meteo returns (dates[] and
    # precipitation_sum[]). Defensive .get(...) chains so a malformed
    # response gracefully degrades to "no data" instead of crashing.
    dates = data.get("daily", {}).get("time", [])
    precip = data.get("daily", {}).get("precipitation_sum", [])

    if not dates:
        return "WEATHER_EVIDENCE: No data returned for the given parameters."

    # Step 4: Aggregate. None values are treated as zero so a single
    # missing day does not poison the total. We also count "days with
    # any rain" because it is a useful summary statistic for a brief.
    records = []
    total_mm = 0.0
    for d, p in zip(dates, precip):
        rain = p if p is not None else 0.0
        total_mm += rain
        records.append({"date": d, "precipitation_mm": rain})

    # Step 5: Build the summary payload. For short ranges we include
    # every daily record. For longer ranges (>14 days) we keep only the
    # first 7 and last 7 days plus the aggregate totals, so the JSON
    # stays small even for "the whole rainy season" style queries.
    summary = {
        "location": {"latitude": latitude, "longitude": longitude},
        "period": {"start": start_date, "end": end_date},
        "total_precipitation_mm": round(total_mm, 2),
        "days_with_rain": sum(1 for r in records if r["precipitation_mm"] > 0),
        "total_days": len(records),
        "daily_records": records if len(records) <= 14 else records[:7] + records[-7:],
    }

    # Stable header prefix so the agent (and the Gradio UI parser) can
    # pick this payload out of the conversation later, exactly like the
    # CASE_BRIEFS_JSON: header used by Tool 4.
    return "WEATHER_EVIDENCE_JSON: " + json.dumps(summary, ensure_ascii=False)


In [ ]:
# Banner so this block is easy to spot when scrolling through the
# notebook output. The // ... // delimiters match the style used in
# the earlier smoke-test cells.
print("// Tool 5: Weather Evidence (optional - weather-related query) //")

# Step 1: Call Tool 5 directly with hard-coded arguments.
# Coordinates are downtown Seattle (47.6062 N, -122.3321 W) and the
# date window is 10 days in January 2023. We are bypassing the agent
# on purpose: the goal is to verify the HTTP call to Open-Meteo and
# the JSON parsing inside get_weather_evidence, NOT the agent's
# ability to extract these arguments from a natural-language prompt.
weather_out = get_weather_evidence(
    latitude=47.6062,
    longitude=-122.3321,
    start_date="2023-01-10",
    end_date="2023-01-20",
)

# Step 2: Tool 5 returns a single string. On success it has the stable
# "WEATHER_EVIDENCE_JSON: {...}" envelope (the parseable handoff the
# agent and the Gradio UI key off later). On failure it returns a
# plain error string with no prefix.
#
# Here we split on that exact prefix, json.loads the payload, and
# pretty-print so a student can eyeball the daily precip / temp rows.
# This mirrors what the agent will do internally when it consumes the tool response.
if weather_out.startswith("WEATHER_EVIDENCE_JSON:"):
    wdata = json.loads(weather_out.replace("WEATHER_EVIDENCE_JSON: ", ""))
    print("\n=== Weather Evidence (pretty) ===")
    print(json.dumps(wdata, indent=2, ensure_ascii=False))
else:
    # Error path: print the raw string so the failure mode is visible.
    print(weather_out)


In [ ]:
# Step 1: Build a fresh Azure OpenAI chat client.
# Same pattern as every other re-assembly cell (Parts 3.9 / 3.12 / 3.14):
# OpenAIChatClient + azure_endpoint routes through your Azure OpenAI
# deployment, `model` is the deployment name from Azure AI Foundry.
client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# Step 2: Rebuild the agent with ALL FIVE tools and a beefier system prompt.
#
# The instructions string is the agent's "house rules":
#   - "Respond with holdings, statutes, issues, and dispositions" tells
#     the LLM what shape the final answer should take (it lines up
#     exactly with the fields case_analyst_extract pulls in Tool 4).
#   - The "If the question involves weather..." sentence reinforces the
#     OPTIONAL trigger that get_weather_evidence already advertises in
#     its @tool description, making it more likely the model invokes
#     Tool 5 on the prompt below.
#   - "Run every tool only once" keeps the tool trace readable for the lab.
agent = client.as_agent(
    instructions=(
        "You are a helpful legal assistant. Respond with holdings, statutes, issues, and dispositions. "
        "If the question involves weather, flooding, storms, precipitation, drainage, or water damage, then call the get_weather_evidence tool. "
        "Run every tool only once."
    ),
    tools=[
        keyword_case_search,
        semantic_case_search,
        precedent_graph_search,
        case_analyst_extract,
        get_weather_evidence,
    ],
)

# Step 3: The flagship user prompt.
# Adds two things versus the earlier runs:
#   - A concrete date ("February of 2024") so get_weather_evidence has
#     a real date range to query the Open-Meteo API for.
#   - An explicit ask for "a structured legal analysis and draft brief"
#     so the LLM composes a memo instead of a bullet list of cases.
user_query = (
    "I have a client in Seattle, Washington, whose property floods after the city "
    "modified road drainage in February of 2024. "    
)

# Step 4: Run the agent. Under the hood Agent Framework will:
#   (a) send the prompt + all FIVE tool schemas to Azure OpenAI,
#   (b) execute each tool call the model requests (printing the
#       "was called" trace lines, plus Tool 4's per-case
#       "extracted_case=..." lines and Tool 5's lat/lon/date echo),
#   (c) feed each tool's output back into the conversation,
#   (d) loop until the model returns a natural-language answer.
print("// Functions the Agent Called: //")
result = await agent.run(user_query)

# Step 5: Render the final answer as Markdown. Expect a draft legal
# brief that quotes real holdings (from Tool 4), cites statutes when
# they surfaced, and weaves the February 2024 rainfall totals (from
# Tool 5) into the factual side of the analysis. That cross-domain
# synthesis is the headline capability of this notebook.
print("")
print("// Agent Response: //")
Markdown(result.text)


In [ ]:
# Stable user id - Mem0 namespaces memories per user, so memories about
# this client never leak into another client's session. Change this to
# a new value to start fresh without dropping the underlying table.
USER_ID = "client-001"

# Mem0 configuration - three pieces:
#   - vector_store: pgvector in HorizonDB (re-using AZURE_PG_* env vars).
#                   This is the headline integration: memories live in
#                   the SAME Postgres as the case rows, AGE graph, and
#                   opinion embeddings.
#   - llm:          Azure OpenAI chat deployment Mem0 uses to EXTRACT
#                   durable facts from each turn (same deployment the
#                   agent itself uses, just wearing a different hat).
#   - embedder:     Azure OpenAI embedding deployment Mem0 uses to
#                   INDEX and SEARCH the stored memories. 1536 dims
#                   matches text-embedding-3-small.
mem0_config = {
    "vector_store": {
        "provider": "pgvector",
        "config": {
            "host":     os.environ["AZURE_PG_HOST"],
            "port":     int(os.environ["AZURE_PG_PORT"]),
            "user":     os.environ["AZURE_PG_USER"],
            "password": os.environ["AZURE_PG_PASSWORD"],
            "dbname":   os.environ["AZURE_PG_NAME"],
            "collection_name":      "counsel_agent_memory",
            "embedding_model_dims": 1536,
        },
    },
    "llm": {
        "provider": "azure_openai",
        "config": {
            "model": os.environ["AZURE_OPENAI_DEPLOYMENT"],
            "azure_kwargs": {
                "azure_endpoint":   os.environ["AZURE_OPENAI_ENDPOINT"],
                "api_key":          os.environ["AZURE_OPENAI_KEY"],
                "api_version":      os.environ["AZURE_API_VERSION"],
                "azure_deployment": os.environ["AZURE_OPENAI_DEPLOYMENT"],
            },
        },
    },
    "embedder": {
        "provider": "azure_openai",
        "config": {
            "model": os.environ["AZURE_EMBED_DEPLOYMENT"],
            "azure_kwargs": {
                "azure_endpoint":   os.environ["AZURE_OPENAI_ENDPOINT"],
                "api_key":          os.environ["AZURE_OPENAI_KEY"],
                "api_version":      os.environ["AZURE_API_VERSION"],
                "azure_deployment": os.environ["AZURE_EMBED_DEPLOYMENT"],
            },
        },
    },
}

# Mem0 will create the `counsel_agent_memory` collection in HorizonDB
# on first use if it does not already exist.
memory = Memory.from_config(mem0_config)


def _memory_search_with_retry(**kw):
    """Call memory.search(**kw) with a short retry loop so a HorizonDB
    forced failover (a few seconds of unavailability) is ridden out instead
    of immediately returning empty memories. Returns the same shape Mem0 does
    ({"results": [...]}). After 5 failed attempts it gives up and returns an
    empty result set so the calling turn can still proceed.
    """
    for attempt in range(1, 6):  # up to 5 attempts, 1s apart
        try:
            return memory.search(**kw)
        except Exception as _e:
            print(f"Mem0: search attempt {attempt}/5 failed ({_e}). Retrying...")
            time.sleep(1)
    print("Mem0: search unavailable after 5 attempts, returning no results.")
    return {"results": []}


def fetch_memory_context(query: str) -> str:
    """Pull the top-k memories most relevant to the upcoming user turn.

    Mem0 embeds `query` with our Azure embedding deployment, runs a
    pgvector similarity search against the `counsel_agent_memory`
    collection scoped to `user_id`, and returns the top-5 hits.
    """
    hits = _memory_search_with_retry(query=query, user_id=USER_ID, limit=5).get("results", [])
    if not hits:
        return "(no prior memories for this user)"
    return "\n".join(f"- {h['memory']}" for h in hits)


def build_counsel_agent(memory_block: str):
    """Re-build the agent each turn so freshly retrieved memories appear in the system prompt.

    The memory block is baked into the system prompt at agent
    construction time, so the cleanest way to expose new memories to
    the LLM is to rebuild the agent. Cheap to do; keeps the wiring
    obvious.
    """
    client = OpenAIChatClient(
        model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_KEY"],
    )
    return client.as_agent(
        instructions=(
            "You are Counsel, an AI legal associate helping analyze Washington State legal cases.\n\n"
            # The "## Long-term memory..." heading + bullet list is the
            # actual mechanism by which Mem0's recall reaches the LLM:
            # it's just text in the system prompt.
            "## Long-term memory about this client (from prior conversations)\n"
            f"{memory_block}\n\n"
            "Use the memory above to personalize answers - refer to the client by name, skip "
            "intake questions you already know the answer to, and build on facts established "
            "in earlier turns.\n\n"
            "When a question requires legal research, use your tools (keyword_case_search, "
            "semantic_case_search, precedent_graph_search, case_analyst_extract, "
            "get_weather_evidence) in that order, each at most once, then synthesize an legal analysis."
        ),
        tools=[
            keyword_case_search,
            semantic_case_search,
            precedent_graph_search,
            case_analyst_extract,
            get_weather_evidence,
        ],
    )


async def run_turn(user_query: str):
    """One conversation turn: retrieve -> run agent -> persist new memories."""
    print(f"\n=== USER ===\n{user_query}\n")

    # Step 1: RETRIEVE - pull the most relevant prior memories for this
    # user, embed-search style, BEFORE the agent runs.
    memory_block = fetch_memory_context(user_query)
    print("--- Memories retrieved for this turn ---")
    print(memory_block)

    # Step 2: RUN - rebuild the agent with the freshly retrieved memory
    # block injected into the system prompt, then run the turn.
    agent = build_counsel_agent(memory_block)
    result = await agent.run(user_query)

    # Step 3: PERSIST - hand both sides of the turn (user prompt +
    # agent response) back to Mem0. Mem0's LLM reads the pair, extracts
    # any new durable facts, and writes them to the pgvector store
    # scoped to USER_ID so the NEXT turn can recall them.
    added = memory.add(
        [
            {"role": "user",      "content": user_query},
            {"role": "assistant", "content": result.text},
        ],
        user_id=USER_ID,
    )
    new_facts = [m["memory"] for m in added.get("results", [])]
    print("\n--- New memories extracted from this turn ---")
    for f in new_facts or ["(none)"]:
        print(f"+ {f}")

    print("\n=== AGENT RESPONSE ===")
    return Markdown(result.text)


# ─────────────────────────────────────────────────────────────────────
# Conversation simulation - WHICHEVER user_query block is LAST in this
# cell wins (Python just re-assigns the same variable). To step through
# the turns, move the comment markers so the turn you want to run is
# the last assignment, then re-execute the cell.
# ─────────────────────────────────────────────────────────────────────

# ── TURN 1: initial intake (run this first) ──────────────────────────

user_query = (
    "Hi, I am an attorney and my client owns a property in the 'Queen Anne neighborhood' of Seattle, Washington."
    "Please remember those details for later. No legal question yet."
)

# ── TURN 2: describe the problem WITHOUT repeating any context ───────
# Notice this prompt never says "Queen Anne" or "Seattle" - the agent
# should pick those up from the memories retrieved at the start of the turn.

#user_query = (
#     "After a heavy storm during February of 2024 my client's yard floods every time it rains. "
#     "Right after the storm, the city regraded the road. "
#     "Does my client have a claim?"
#)

#── TURN 3: follow-up that depends on TURN 2's analysis ──────────────
# "Given what you found" only makes sense if the agent remembers the
# legal theory it landed on in Turn 2 - that's the memory at work.

#user_query = (
#     "Given what you found, what "
#     "evidence should I start collecting?"
#)

# ── TURN 4: verify long-term recall ──────────────────────────────────
# Pure recall test - the agent should answer from memory alone, no
# retrieval tools needed.

#user_query = (
#     "Remind me - what neighborhood is my client's property in, and what was the month of the storm we discussed?"
#)

await run_turn(user_query)


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Counsel AI - Gradio web UI for the full 5-tool + Mem0 agent
# ─────────────────────────────────────────────────────────────────────
#
# This cell is long but laid out in clear sections - read it top to
# bottom and you'll see the same building blocks from the rest of the
# notebook, just plumbed into a Gradio app:
#
#   1) Create the Counsel agent for the UI (OpenAIChatClient + as_agent
#      with all 5 tools - same pattern as earlier).
#   2) Wire up Mem0 long-term memory under a UI-only user id, wiping
#      that user's memories on every cell run so the app starts fresh.
#   3) Helper functions to retrieve / format memories before each turn
#      and to extract / persist new memories after each turn (same
#      retrieve -> run -> persist lifecycle).
#   4) A streaming chat handler that runs the agent, captures the
#      print("<tool> was called") trace lines from each tool, and
#      feeds them into the Tool Trace panel as they happen.
#   5) gr.Blocks layout - chat on the left, Tool Trace + Long-term
#      Memory panels on the right.
#   6) demo.launch(server_port=7860, inline=False) - starts the local
#      web server. Open http://localhost:7860 in your browser AFTER
#      this cell prints "Running on local URL: http://127.0.0.1:7860".
# ─────────────────────────────────────────────────────────────────────

# ── 1) Create the Counsel Agent for the UI ──────────────────────────

_ui_client = OpenAIChatClient(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
)

# ── Mem0 long-term memory for the UI session ────────────────────────
#
# We re-use the same `memory` client (pgvector in HorizonDB + Azure OpenAI)
# initialised in the Mem0 demo cell above. The UI runs under its own user
# id so it stays isolated from the demo's USER_ID, and we wipe that user's
# memories every time this cell is executed so the launched app always
# starts from a clean slate.
_UI_USER_ID = "ui-counsel-session2"

try:
    memory.delete_all(user_id=_UI_USER_ID)
    print(f"Mem0: cleared prior memories for user '{_UI_USER_ID}'.")
except Exception as _e:
    print(f"Mem0: nothing to clear for user '{_UI_USER_ID}' ({_e}).")


_UI_TOOLS = [
    keyword_case_search,
    semantic_case_search,
    precedent_graph_search,
    case_analyst_extract,
    get_weather_evidence,
]


def _fetch_ui_memory_list(query: str) -> list[str]:
    """Pull the top-k memories Mem0 considers most relevant to the upcoming user turn."""
    hits = _memory_search_with_retry(query=query, user_id=_UI_USER_ID, limit=5).get("results", [])
    return [h["memory"] for h in hits]


def _format_memory_block(items: list[str]) -> str:
    """Render a list of memories into the bullet block injected into the system prompt."""
    if not items:
        return "(no prior memories yet)"
    return "\n".join(f"- {m}" for m in items)


def _build_ui_agent(memory_block: str):
    """Re-build the agent each turn so freshly retrieved memories appear in the system prompt."""
    return _ui_client.as_agent(
        instructions=(
            "You are Counsel, an AI legal associate helping analyze Washington State legal cases.\n\n"
            "## Long-term memory about this client (from prior conversations)\n"
            f"{memory_block}\n\n"
            "Use the memory above to personalize answers - refer to the client by name, skip "
            "intake questions you already know the answer to, and build on facts established "
            "in earlier turns.\n\n"
            
            "If a legal question, this is the required wWorkflow (if a legal question, always run these 4 tools in order):\n"
            "1. keyword_case_search - BM25 full-text search for canonical cases\n"
            "2. semantic_case_search - DiskANN vector search for factually similar cases\n"
            "3. precedent_graph_search - Citation graph traversal via Apache AGE\n"
            "4. case_analyst_extract - Extract structured legal facts via azure_ai.extract\n\n"

            "Weather / Storm / Flooding Tool:\n"
            "5. Always run this tool last if ran. If the question involves weather, flooding, storms, precipitation, drainage, or water damage,\n"
            "use get_weather_evidence to retrieve historical precipitation data for the relevant location and date.\n"
            "Include this data as supporting evidence in your analysis.\n\n"

            "Constraints:\n"
            "- If a legal question, do NOT skip the required steps (1-4).\n"
            "- If any mention of flooding, storms, drainage, or water damage is detected, you must use the Weather / Storm / Flooding Tool. And run this tool last.\n"
            "- Pass outputs between tools (case ids, JSON payloads).\n"
            "- If a legal question, do NOT manually summarize without tools.\n"
            "- If a legal question, call each tool AT MOST ONCE. Never call the same tool twice.\n\n"            

            "If this is a legal question, after all tool calls complete, synthesize the results into a polished legal analysis.\n"
            "- If weather evidence was gathered, a paragraph incorporating precipitation data as supporting evidence.\n"
            "- **Key Takeaway:** A concise statement of the principal legal rule.\n"
            "- **Key Cases in this Line of Authority:** A bulleted list of relevant cases, each with case name, citation, and year.\n"
            "- **Statutes Extracted:** A bulleted list of statutes and what case it is from.\n"
            "- **Holdings Extracted:** A bulleted list of holdings and what case it is from.\n"
        ),
        tools=_UI_TOOLS,
    )

# ── Tool Trace Metadata ──────────────────────────────────────────────
_TOOL_META = {
    "keyword_case_search":    {"idx": 1, "name": "KeywordCaseSearchTool",  "tech": "BM25 Search"},
    "semantic_case_search":   {"idx": 2, "name": "SemanticCaseSearchTool", "tech": "DiskANN with Advanced Filtering"},
    "precedent_graph_search": {"idx": 3, "name": "PrecedentGraphTool",     "tech": "AGE Graph"},
    "case_analyst_extract":   {"idx": 4, "name": "CaseAnalystTool",        "tech": "AI Function azure_ai.extract()"},
    "get_weather_evidence":   {"idx": 5, "name": "WeatherEvidenceTool",    "tech": "Weather External API"},
}

_TRACE_PALETTE = {
    "accent": "#5b5ef7",
    "accent_soft": "#eef0ff",
    "accent_border": "#d8ddff",
    "text": "#1f2340",
    "muted": "#687087",
    "panel": "#ffffff",
    "line": "#dfe3f0",
    "success": "#217346",
    "success_soft": "#edf8f0",
    "success_border": "#cce8d5",
    "success_text": "#1f5f3c",
    "shadow": "0 10px 28px rgba(31, 35, 64, 0.08)",
}

_UI_THEME = gr.themes.Soft(
    primary_hue="indigo",
    neutral_hue="gray",
    radius_size=gr.themes.sizes.radius_lg,
)


def _tool_icon_svg(key: str) -> str:
    accent = _TRACE_PALETTE["accent"]
    icons = {
        "keyword_case_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<circle cx="8.5" cy="8.5" r="5.5" stroke="{accent}" stroke-width="1.8"/>'
            f'<path d="M12.5 12.5L16.5 16.5" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "semantic_case_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<ellipse cx="10" cy="5" rx="5.8" ry="2.6" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M4.2 5V10.2C4.2 11.6 6.8 12.8 10 12.8C13.2 12.8 15.8 11.6 15.8 10.2V5" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M4.2 10.2V15C4.2 16.4 6.8 17.6 10 17.6C13.2 17.6 15.8 16.4 15.8 15V10.2" stroke="{accent}" stroke-width="1.6"/>'
            f'</svg>'
        ),
        "precedent_graph_search": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<circle cx="5" cy="10" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<circle cx="15" cy="5" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<circle cx="15" cy="15" r="1.7" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M6.5 9L13.2 5.8" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M6.5 11L13.2 14.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M15 6.8V13.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "case_analyst_extract": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<path d="M6 2.8H11.8L15.5 6.5V16.5C15.5 17.3 14.8 18 14 18H6C5.2 18 4.5 17.3 4.5 16.5V4.3C4.5 3.5 5.2 2.8 6 2.8Z" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M11.5 2.8V6.4H15.2" stroke="{accent}" stroke-width="1.6"/>'
            f'<path d="M7.5 10H12.5" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M7.5 13H11.2" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
        "get_weather_evidence": (
            f'<svg width="18" height="18" viewBox="0 0 20 20" fill="none" '
            f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
            f'<path d="M6 14C3.8 14 2 12.4 2 10.4C2 8.7 3.3 7.3 5 7C5.5 4.7 7.5 3 10 3C12.8 3 15 5 15.3 7.5C17 7.8 18 9.2 18 10.8C18 12.6 16.5 14 14.7 14H6Z" stroke="{accent}" stroke-width="1.6" stroke-linejoin="round"/>'
            f'<path d="M8 16V14" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M12 17V14" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'<path d="M10 18V15" stroke="{accent}" stroke-width="1.6" stroke-linecap="round"/>'
            f'</svg>'
        ),
    }
    return icons.get(key, "")


def _header_icon_svg() -> str:
    accent = _TRACE_PALETTE["accent"]
    return (
        f'<svg width="28" height="28" viewBox="0 0 24 24" fill="none" '
        f'xmlns="http://www.w3.org/2000/svg" aria-hidden="true">'
        f'<path d="M4 7H20" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'<path d="M12 5V19" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'<path d="M7 7C7 10 5.5 12 4 13.3C5 14.1 6.2 14.5 7.5 14.5C9.4 14.5 10.9 13.4 12 11.8" stroke="{accent}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>'
        f'<path d="M17 7C17 10 18.5 12 20 13.3C19 14.1 17.8 14.5 16.5 14.5C14.6 14.5 13.1 13.4 12 11.8" stroke="{accent}" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>'
        f'<path d="M9 19H15" stroke="{accent}" stroke-width="1.8" stroke-linecap="round"/>'
        f'</svg>'
    )


# ── Trace Parsing Helpers ────────────────────────────────────────────

def _parse_traces(captured: str) -> list[dict]:
    """Parse captured stdout to identify which tools the agent called."""
    traces, lines = [], captured.split("\n")
    for i, line in enumerate(lines):
        for key, meta in _TOOL_META.items():
            if f"{key} was called" in line:
                details = []
                for nxt in lines[i + 1:]:
                    if nxt.startswith("  "):
                        details.append(nxt.strip())
                    else:
                        break
                traces.append({"key": key, "details": details, **meta})
                break
    return traces


def _new_trace_state() -> dict:
    return {
        "traces": [],
        "line_buffer": "",
        "changed": False,
    }


def _ingest_trace_chunk(state: dict, chunk: str, now: float | None = None) -> None:
    """Update trace state as stdout arrives so the UI stays in sync with actual tool starts."""
    if not chunk:
        return

    if now is None:
        now = time.perf_counter()

    state["line_buffer"] += chunk
    while "\n" in state["line_buffer"]:
        line, state["line_buffer"] = state["line_buffer"].split("\n", 1)
        stripped = line.rstrip("\r")

        matched_key = None
        for key in _TOOL_META:
            if f"{key} was called" in stripped:
                matched_key = key
                break

        if matched_key is not None:
            trace = {"key": matched_key, "details": [], "started_at": now, **_TOOL_META[matched_key]}
            state["traces"].append(trace)
            state["changed"] = True
            continue

        if stripped.startswith("  ") and state["traces"]:
            state["traces"][-1]["details"].append(stripped.strip())
            state["changed"] = True


def _run_ui_agent(message: str, memory_block: str):
    """Build an agent with the latest Mem0 context, then run it on a worker thread."""
    agent = _build_ui_agent(memory_block)
    return asyncio.run(agent.run(message))


def _trace_desc(t: dict) -> str:
    d = " ".join(t.get("details", []))
    k = t["key"]
    if k == "keyword_case_search":
        m = re.search(r"query=(.+?)\s\s+court_level=", d)
        if m:
            q = m.group(1).strip().strip("'\"").replace("\\'", "'").replace('\\"', '"')
        else:
            q = "case law terms"
        q = q[:55]
        return f'Searched for "{q}" and related terms.'
    if k == "semantic_case_search":
        parts = []
        m = re.search(r"court_level=([^\s]+)", d)
        if m and m.group(1) != "None":
            parts.append(f"court_level IN ({m.group(1)})")
        m = re.search(r"min_year=(\d+)", d)
        if m:
            parts.append(f"decision_date >= {m.group(1)}")
        filt = ", ".join(parts) if parts else "no additional filters"
        return f"Expanded search using vector similarity with filters: {filt}."
    if k == "precedent_graph_search":
        m = re.search(r"inputs=(\d+)", d)
        n = m.group(1) if m else "N"
        return f"{n} unique cases found via KeywordCaseSearchTool and SemanticCaseSearchTool."
    if k == "case_analyst_extract":
        fm = re.search(r"fields=([^\s]+)", d)
        cm = re.search(r"case_ids=(\d+)", d)
        n_in = cm.group(1) if cm else "the"
        if fm:
            field_list = [f.strip() for f in fm.group(1).split(",") if f.strip()]
            chips = "".join(
                f'<span style="display:inline-block;padding:2px 8px;margin:2px 4px 2px 0;'
                f'background:#eef0ff;border:1px solid #d8ddff;border-radius:999px;'
                f'font-size:11px;color:#4f56c7;">{f}</span>'
                for f in field_list
            )
            return (
                f'Extracting structured legal entities from {n_in} case opinions:'
                f'<div style="margin-top:6px;">{chips}</div>'
            )
        return f"Extracting structured fields from {n_in} key opinions."
    if k == "get_weather_evidence":
        m = re.search(r"start=(\S+)", d)
        dt = m.group(1) if m else "requested dates"
        return f"Retrieved historical precipitation data for {dt}."
    return ""


def _trace_result(t: dict) -> str:
    d = " ".join(t.get("details", []))
    k = t["key"]
    if k == "keyword_case_search":
        return "Results: 5 cases"
    if k == "semantic_case_search":
        return "Results: 5 cases"
    if k == "precedent_graph_search":
        rm = re.search(r"related_count=(\d+)", d)
        hm = re.search(r"hops=(\d+)", d)
        if rm:
            n = rm.group(1)
            hops = hm.group(1) if hm else "2"
            return f"{n} additional related cases found traversing the graph using {hops} hops"
        return "Paths found via graph traversal"
    if k == "case_analyst_extract":
        cases = re.findall(r"extracted_case=(\d+)\|(.+?)(?=\s+extracted_case=|\s+\w+=|$)", d)
        if cases:
            items = "".join(
                f'<div style="display:flex;align-items:flex-start;gap:6px;margin-top:4px;">'
                f'<span style="color:#217346;font-weight:700;">✓</span>'
                f'<span style="font-size:12px;color:#1f2340;">{name.strip()} '
                f'<span style="color:#687087;">(id {cid})</span></span>'
                f'</div>'
                for cid, name in cases
            )
            return (
                f'<div style="font-weight:600;color:#1f5f3c;margin-top:4px;">'
                f'Extracted {len(cases)} case{"s" if len(cases) != 1 else ""}:</div>'
                f'{items}'
            )
        m = re.search(r"case_ids=(\d+)", d)
        return f"Cases analyzed: {m.group(1) if m else 'N'}"
    return ""


# ── Trace Panel HTML Builder (Light Theme + Flat Icons) ─────────────

def _build_trace_html(traces, t0, running=False, total_time=None):
    """Build trace panel HTML. Supports live updates during streaming."""
    panel = _TRACE_PALETTE["panel"]
    text = _TRACE_PALETTE["text"]
    muted = _TRACE_PALETTE["muted"]
    accent = _TRACE_PALETTE["accent"]
    accent_soft = _TRACE_PALETTE["accent_soft"]
    accent_border = _TRACE_PALETTE["accent_border"]
    line = _TRACE_PALETTE["line"]
    success = _TRACE_PALETTE["success"]
    success_soft = _TRACE_PALETTE["success_soft"]
    success_border = _TRACE_PALETTE["success_border"]
    success_text = _TRACE_PALETTE["success_text"]
    shadow = _TRACE_PALETTE["shadow"]
    now = time.perf_counter()

    if not traces and not running:
        return (
            f'<div style="padding:24px 22px;text-align:center;color:{muted};background:{panel};">'
            f'<div style="width:52px;height:52px;border-radius:16px;background:{accent_soft};margin:0 auto 12px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:22px;">⌕</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Tool Trace</div>'
            f'<div>Ask a question to see the research steps</div>'
            f'</div>'
        )
    if not traces and running:
        return (
            f'<div style="padding:24px 22px;text-align:center;color:{muted};background:{panel};">'
            f'<div style="width:52px;height:52px;border-radius:16px;background:{accent_soft};margin:0 auto 12px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:22px;" class="spin">◌</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Starting analysis pipeline</div>'
            f'<div>Preparing the first tool call</div>'
            f'</div>'
        )

    n_label = f'{len(traces)} tool{"s" if len(traces) != 1 else ""}'
    state = "running" if running else "used"
    html = (
        f'<div style="background:{panel};padding:2px 2px 8px 2px;">'
        f'<div style="font-weight:700;font-size:15px;color:{text};margin-bottom:14px;">'
        f'Tool Trace <span style="font-weight:500;color:{muted};">({n_label} {state})</span></div>'
    )

    for i, t in enumerate(traces):
        is_last = i == len(traces) - 1
        is_active = running and is_last

        started_at = t.get("started_at", t0)
        if is_active:
            dur = now - started_at
        elif i + 1 < len(traces):
            dur = traces[i + 1].get("started_at", started_at) - started_at
        else:
            end_at = t0 + (total_time or 0)
            dur = end_at - started_at
        dur = max(dur, 0)

        ts = f"{int(dur * 1000)} ms" if dur < 1 else f"{dur:.1f}s"
        desc = _trace_desc(t)
        res = _trace_result(t)
        icon = _tool_icon_svg(t["key"])

        if is_active:
            status_icon = f'<span style="color:{accent};font-size:14px;" class="spin">◌</span>'
            row_bg = "#f8f9ff"
        else:
            status_icon = f'<span style="color:{success};font-size:14px;">●</span>'
            row_bg = panel

        connector = ""
        if i < len(traces) - 1:
            connector = (
                f'<div style="position:absolute;left:15px;top:34px;width:2px;height:40px;'
                f'background:{accent_border};border-radius:999px;"></div>'
            )

        res_div = (
            f'<div style="color:{muted};font-size:13px;margin-top:6px;">{res}</div>'
            if res and (not is_active or t["key"] == "case_analyst_extract") else ""
        )

        html += (
            f'<div style="display:flex;gap:12px;position:relative;margin-bottom:14px;">'
            f'<div style="position:relative;width:32px;flex:0 0 32px;display:flex;justify-content:center;">'
            f'<div style="width:30px;height:30px;border-radius:999px;background:{accent};color:#fff;' 
            f'display:flex;align-items:center;justify-content:center;font-weight:700;font-size:13px;">{t["idx"]}</div>'
            f'{connector}</div>'
            f'<div style="flex:1;background:{row_bg};border:1px solid {line};border-radius:14px;padding:12px 14px;'
            f'box-shadow:{shadow};">'
            f'<div style="display:flex;justify-content:space-between;align-items:flex-start;gap:12px;">'
            f'<div style="display:flex;gap:10px;align-items:flex-start;min-width:0;">'
            f'<div style="width:20px;height:20px;display:flex;align-items:center;justify-content:center;flex:0 0 20px;">{icon}</div>'
            f'<div style="min-width:0;">'
            f'<div style="font-weight:700;color:{text};line-height:1.25;">{t["name"]} '
            f'<span style="font-weight:500;color:{muted};font-size:12px;">({t["tech"]})</span></div>'
            f'<div style="color:{muted};font-size:13px;line-height:1.45;margin-top:4px;">{desc}</div>'
            f'{res_div}</div></div>'
            f'<div style="text-align:right;white-space:nowrap;color:{muted};font-size:13px;">{ts}<div style="margin-top:4px;">{status_icon}</div></div>'
            f'</div></div></div>'
        )

    if running:
        elapsed = now - t0
        html += (
            f'<div style="display:flex;align-items:center;gap:8px;padding:12px 14px;border-radius:12px;' 
            f'background:{accent_soft};border:1px solid {accent_border};">'
            f'<span class="spin" style="color:{text};">◌</span>'
            f'<span style="font-weight:700;color:{text};">Running... {elapsed:.1f}s elapsed</span></div>'
        )
    else:
        t_disp = total_time or 0
        html += (
            f'<div style="display:flex;align-items:center;gap:8px;padding:12px 14px;border-radius:12px;' 
            f'background:{success_soft};border:1px solid {success_border};color:{success_text};">'
            f'<span style="font-size:14px;color:{success};"></span>'
            f'<span style="font-weight:700;color:{success_text};">Completed in {t_disp:.2f} seconds</span></div>'
        )

    html += "</div>"
    return html


# ── Memory Panel HTML Builder ────────────────────────────────────────

def _build_memory_html(retrieved: list[str], new_facts: list[str], capturing: bool = False) -> str:
    """Render the Mem0 side panel showing memories retrieved + captured this turn."""
    text = _TRACE_PALETTE["text"]
    muted = _TRACE_PALETTE["muted"]
    accent = _TRACE_PALETTE["accent"]
    accent_soft = _TRACE_PALETTE["accent_soft"]
    accent_border = _TRACE_PALETTE["accent_border"]
    line = _TRACE_PALETTE["line"]
    success = _TRACE_PALETTE["success"]
    success_soft = _TRACE_PALETTE["success_soft"]
    success_border = _TRACE_PALETTE["success_border"]
    success_text = _TRACE_PALETTE["success_text"]

    # Empty state - shown on first render before any turn has happened.
    if not retrieved and not new_facts and not capturing:
        return (
            f'<div style="padding:6px 0;text-align:center;color:{muted};">'
            f'<div style="width:42px;height:42px;border-radius:14px;background:{accent_soft};margin:0 auto 10px;'
            f'display:flex;align-items:center;justify-content:center;color:{accent};font-size:20px;">◇</div>'
            f'<div style="font-weight:600;color:{text};margin-bottom:4px;">Mem0 Long-Term Memory</div>'
            f'<div style="font-size:13px;">Memories retrieved and captured will appear here each turn.</div>'
            f'</div>'
        )

    html = (
        f'<div style="font-weight:700;font-size:15px;color:{text};margin-bottom:12px;">'
        f'Mem0 Long-Term Memory '
        f'<span style="font-weight:500;color:{muted};font-size:12px;">(user: {_UI_USER_ID})</span>'
        f'</div>'
    )

    # Section 1: memories retrieved for this turn
    html += (
        f'<div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.04em;'
        f'color:{accent};margin:4px 0 8px;">Retrieved for this turn ({len(retrieved)})</div>'
    )
    if retrieved:
        for m in retrieved:
            html += (
                f'<div style="padding:8px 10px;background:{accent_soft};border:1px solid {accent_border};'
                f'border-radius:10px;margin-bottom:6px;font-size:13px;color:{text};line-height:1.4;">'
                f'<span style="color:{accent};font-weight:700;margin-right:6px;">↳</span>{m}</div>'
            )
    else:
        html += (
            f'<div style="padding:8px 10px;background:#f7f8fc;border:1px dashed {line};border-radius:10px;'
            f'font-size:13px;color:{muted};">(no prior memories yet)</div>'
        )

    # Section 2: memories captured from this turn
    if capturing:
        html += (
            f'<div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.04em;'
            f'color:{muted};margin:14px 0 8px;display:flex;align-items:center;gap:6px;">'
            f'<span class="spin" style="color:{muted};">◌</span> Capturing new memories...</div>'
            f'<div style="padding:8px 10px;background:#f7f8fc;border:1px dashed {line};border-radius:10px;'
            f'font-size:13px;color:{muted};">Waiting for the agent to finish before Mem0 extracts facts.</div>'
        )
    else:
        html += (
            f'<div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.04em;'
            f'color:{success};margin:14px 0 8px;">Captured this turn ({len(new_facts)})</div>'
        )
        if new_facts:
            for m in new_facts:
                html += (
                    f'<div style="padding:8px 10px;background:{success_soft};border:1px solid {success_border};'
                    f'border-radius:10px;margin-bottom:6px;font-size:13px;color:{success_text};line-height:1.4;">'
                    f'<span style="color:{success};font-weight:700;margin-right:6px;">+</span>{m}</div>'
                )
        else:
            html += (
                f'<div style="padding:8px 10px;background:#f7f8fc;border:1px dashed {line};border-radius:10px;'
                f'font-size:13px;color:{muted};">(no new facts extracted from this turn)</div>'
            )

    return html


# ── Chat Handler (async generator for live streaming) ────────────────

async def _chat_fn(message: str, history: list):
    if not message or not message.strip():
        yield history, _build_trace_html([], 0), _build_memory_html([], []), ""
        return

    # Mem0: retrieve relevant memories BEFORE we start tee'ing stdout so the
    # tool-trace parser doesn't see these lines. The memory block is injected
    # into the agent's system prompt for this turn.
    retrieved = _fetch_ui_memory_list(message)
    memory_block = _format_memory_block(retrieved)
    print("\n--- Mem0: memories retrieved for this turn ---")
    print(memory_block)

    old_stdout = sys.stdout
    buf = io.StringIO()
    trace_state = _new_trace_state()

    class _Tee:
        def write(self, s):
            buf.write(s)
            _ingest_trace_chunk(trace_state, s)
            old_stdout.write(s)
        def flush(self):
            buf.flush()
            old_stdout.flush()
        def __getattr__(self, name):
            return getattr(old_stdout, name)
    sys.stdout = _Tee()

    t0 = time.perf_counter()
    pending_history = history + [{"role": "user", "content": message}]
    # Show retrieved memories in the side panel immediately; new-memory
    # section is left in "capturing..." state until the agent finishes.
    memory_html = _build_memory_html(retrieved, [], capturing=True)
    yield pending_history, _build_trace_html([], t0, running=True), memory_html, ""

    task = asyncio.create_task(asyncio.to_thread(_run_ui_agent, message, memory_block))
    last_render = 0.0

    while not task.done():
        await asyncio.sleep(0.15)
        now = time.perf_counter()
        should_render = trace_state["changed"] or trace_state["traces"] or (now - last_render >= 0.5)
        if not should_render:
            continue

        trace_state["changed"] = False
        last_render = now
        trace_html = _build_trace_html(trace_state["traces"], t0, running=True)
        yield pending_history, trace_html, memory_html, ""

    elapsed = time.perf_counter() - t0
    sys.stdout = old_stdout

    try:
        res = task.result()
        response = res.text
    except Exception as e:
        response = f"⚠️ An error occurred: {e}"

    # Mem0: persist this turn so the next message can recall it. Done after
    # stdout is restored so the prints go straight to the notebook output
    # (and not into the tool-trace buffer).
    new_facts: list[str] = []
    if not response.startswith("⚠️"):
        try:
            added = memory.add(
                [
                    {"role": "user",      "content": message},
                    {"role": "assistant", "content": response},
                ],
                user_id=_UI_USER_ID,
            )
            new_facts = [m["memory"] for m in added.get("results", [])]
            print("--- Mem0: new memories extracted from this turn ---")
            for f in new_facts or ["(none)"]:
                print(f"+ {f}")
        except Exception as _e:
            print(f"Mem0: failed to persist this turn ({_e}).")

    parsed = _parse_traces(buf.getvalue())
    if len(parsed) > len(trace_state["traces"]):
        for i, parsed_trace in enumerate(parsed):
            if i < len(trace_state["traces"]):
                continue
            parsed_trace["started_at"] = t0 + elapsed
            trace_state["traces"].append(parsed_trace)

    trace_html = _build_trace_html(trace_state["traces"], t0, running=False, total_time=elapsed)
    memory_html = _build_memory_html(retrieved, new_facts, capturing=False)

    final_history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response},
    ]
    yield final_history, trace_html, memory_html, ""


# ── Status Bar ───────────────────────────────────────────────────────

def _status_html() -> str:
    now = datetime.now().strftime("%I:%M %p")
    text = _TRACE_PALETTE["muted"]
    success = _TRACE_PALETTE["success"]
    return (
        f'<div style="display:flex;align-items:center;gap:6px;padding:8px 16px;'
        f'font-size:12px;color:{text};flex-wrap:wrap;">'
        f'Connected to HorizonDB (Reader Endpoint) '
        f'<span style="color:{success};">●</span>'
        f'<span style="margin:0 6px;">|</span> Database: {DB_CONFIG.get("dbname", "")}'
        f'<span style="margin:0 6px;">|</span> User: {DB_CONFIG.get("user", "")}'
        f'<span style="margin:0 6px;">|</span> Time: {now}</div>'
    )


# ── Sample Prompts ───────────────────────────────────────────────────
# These mirror the four TURN prompts from the Mem0 demo cell above, so
# students can click through the same conversation flow inside the UI and
# watch the memory panel light up turn by turn. The 5th prompt is a
# deliberately silly one to show how the agent handles novel fact patterns.

_PROMPT_1 = (
    "Hi, I am an attorney and my client owns a property in the 'Queen Anne neighborhood' of Seattle, Washington. "
    "Please remember those details for later. No legal question yet."
)
_PROMPT_2 = (
    "After a heavy storm during April and May of 2024, my client's yard now floods every time it rains. "
    "The city repaved the road right before the storm. "
    "Does my client have a claim?"
)
_PROMPT_3 = (
    "Given what you found, what "
    "evidence should I start collecting?"
)
_PROMPT_4 = (
    "I have a second client, whose condo in downtown Seattle, Washington had a spaceship crash into it, "
    "and also destroy the traffic lights on the street. What claims does my client have?"
)


# ── Build the Gradio App ────────────────────────────────────────────

_CSS = """
:root {
    --counsel-bg: #f6f7fb;
    --counsel-panel: #ffffff;
    --counsel-line: #e2e7f3;
    --counsel-text: #1f2340;
    --counsel-muted: #687087;
    --counsel-accent: #5b5ef7;
    --counsel-accent-soft: #eef0ff;
    --counsel-font: Aptos, "Segoe UI", Tahoma, Geneva, Verdana, sans-serif;
}

.gradio-container,
.gradio-container * {
    font-family: var(--counsel-font) !important;
}

.gradio-container {
    background: linear-gradient(180deg, #fbfbfe 0%, #f4f6fb 100%);
}

.counsel-shell {
    background: transparent;
}

.counsel-hdr {
    padding: 2px 2px 10px 2px;
}

.counsel-hdr h1 {
    margin: 0;
    font-size: 24px;
    display: flex;
    align-items: center;
    gap: 10px;
    color: var(--counsel-text);
}

.counsel-hdr p {
    margin: 4px 0 0;
    font-size: 14px;
    color: var(--counsel-muted);
}

.counsel-header-icon {
    width: 30px;
    height: 30px;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    flex: 0 0 30px;
}

.counsel-chat,
.counsel-trace,
.counsel-memory {
    border: 1px solid var(--counsel-line);
    border-radius: 18px;
    background: var(--counsel-panel);
    box-shadow: 0 16px 34px rgba(31, 35, 64, 0.06);
}

.counsel-chat {
    overflow: hidden;
}

.counsel-trace,
.counsel-memory {
    padding: 18px;
}

.counsel-memory {
    margin-top: 12px;
}

.counsel-right-stack {
    background: transparent;
    border: 0;
    box-shadow: none;
    padding: 0;
}

.counsel-chatbot,
.counsel-chatbot > .wrap,
.counsel-chatbot .wrap,
.counsel-chatbot .bubble-wrap,
.counsel-chatbot .message-wrap,
.counsel-chatbot .message-row,
.counsel-chatbot .panel,
.counsel-chatbot .scroll-hide {
    background: #ffffff !important;
}

.counsel-chatbot {
    border: 0 !important;
}

.counsel-chatbot .message,
.counsel-chatbot .message.user,
.counsel-chatbot .message.bot,
.counsel-chatbot .bubble {
    background: transparent !important;
    color: var(--counsel-text) !important;
    border: 0 !important;
    box-shadow: none !important;
}

.counsel-chatbot .message-content {
    background: #ffffff !important;
    color: var(--counsel-text) !important;
    border: 1px solid var(--counsel-line) !important;
    box-shadow: none !important;
}

.counsel-chatbot .message.user {
    background: #f4f6ff !important;
}

.counsel-chatbot .message.bot {
    background: #ffffff !important;
}

.counsel-chatbot .message * {
    color: var(--counsel-text) !important;
}

.counsel-chatbot .avatar-container,
.counsel-chatbot .avatar-container * {
    background: #f4f6ff !important;
    color: var(--counsel-accent) !important;
}

.counsel-examples {
    align-items: center;
    gap: 8px;
    flex-wrap: wrap;
}

.counsel-examples .gr-button {
    border: 1px solid #dde2ff;
    background: #f7f8ff;
    color: #4f56c7;
    border-radius: 10px;
}

.counsel-input-row {
    gap: 12px;
    align-items: center;
}

.counsel-input,
.counsel-input > div,
.counsel-input .wrap,
.counsel-input .block {
    background: #ffffff !important;
    border-radius: 14px !important;
}

.counsel-input textarea,
.counsel-input input,
.counsel-input-row textarea,
.counsel-input-row input {
    background: #ffffff !important;
    border: 1px solid var(--counsel-line) !important;
    border-radius: 14px !important;
    color: var(--counsel-text) !important;
    box-shadow: none !important;
}

.counsel-input textarea::placeholder,
.counsel-input input::placeholder {
    color: #8f96ab !important;
}

.counsel-send {
    border-radius: 14px !important;
}

footer { display: none !important; }
@keyframes spin { from { transform: rotate(0deg); } to { transform: rotate(360deg); } }
.spin { display: inline-block; animation: spin 1s linear infinite; }
"""

with gr.Blocks(title="Counsel AI – Legal Research Assistant", css=_CSS, theme=_UI_THEME) as demo:
    with gr.Column(elem_classes=["counsel-shell"]):
        gr.HTML(
            '<div class="counsel-hdr">'
            f'<h1><span class="counsel-header-icon">{_header_icon_svg()}</span> Counsel AI - Legal Research Assistant</h1>'
            '<p>Ask legal questions. Counsel uses HorizonDB to find, analyze, and synthesize case law.</p>'
            '</div>'
        )

        with gr.Row(equal_height=True):
            with gr.Column(scale=3, elem_classes=["counsel-chat"]):
                chatbot = gr.Chatbot(height=500, show_label=False, type="messages", elem_classes=["counsel-chatbot"])
            with gr.Column(scale=2, elem_classes=["counsel-right-stack"]):
                with gr.Column(elem_classes=["counsel-trace"]):
                    trace_panel = gr.HTML(value=_build_trace_html([], 0))
                with gr.Column(elem_classes=["counsel-memory"]):
                    memory_panel = gr.HTML(value=_build_memory_html([], []))

        with gr.Row(elem_classes=["counsel-examples"]):
            gr.HTML('<span style="font-size:13px;color:#687087;padding:0 4px;">Example questions</span>')
            ex1 = gr.Button("1. Queen Anne client", size="sm")
            ex2 = gr.Button("2. Yard flooding after neighbor regrade & city repave", size="sm")
            ex3 = gr.Button("3. Most promising defendant & evidence to collect", size="sm")            
            ex4 = gr.Button("4. Spaceship crashed into a second client's condo", size="sm")

        with gr.Row(elem_classes=["counsel-input-row"]):
            msg = gr.Textbox(
                placeholder="Ask a legal question...",
                show_label=False,
                scale=5,
                container=False,
                elem_classes=["counsel-input"],
            )
            send_btn = gr.Button("Send", variant="primary", scale=1, elem_classes=["counsel-send"])

        gr.HTML(_status_html())

    send_btn.click(_chat_fn, [msg, chatbot], [chatbot, trace_panel, memory_panel, msg])
    msg.submit(_chat_fn, [msg, chatbot], [chatbot, trace_panel, memory_panel, msg])

    ex1.click(lambda: _PROMPT_1, outputs=msg)
    ex2.click(lambda: _PROMPT_2, outputs=msg)
    ex3.click(lambda: _PROMPT_3, outputs=msg)
    ex4.click(lambda: _PROMPT_4, outputs=msg)    

gr.close_all()
demo.launch(server_port=7860, inline=False)
